# M6L1 - Review of Computing Concepts

In this Notebook, we will review the concepts covered in this course so that you can apply them outside of Jupyter Notebook for your own use. We have focused on teaching a practical application of RNA-Seq techniques within the Jupyter Notebook format, but the majority of the time, researchers will use software on their local devices or within a cluster. This saves time, and there are many tools to help with the formatting and syntax of code!

### **This notebook covers:**
- Lecture on an overview of computing options for the HPC cluster
- RSS Ondemand - Hellbender Desktop, Code Server, and SSH connections (via Putty, the Terminal, and Powershell)
- Terminal usage of the Hellbender Cluster - SLURM, SBatch, and Terminal via SSH Connections
- Environment management and Lmod usage within the cluster


```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

<iframe width="560" height="315" src="https://www.youtube.com/embed/RqSeNNWB_NA?si=wPSp55NWPhhBhGWG" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

## RSS OnDemand 
[RSSOndemand](https://hb-classes.missouri.edu) provides access to:
- **Hellbender Desktop**: graphical interface on the cluster
- **Code Server**: browser-based VS Code
- **RStudio**: browser-based R environment
- **Jupyter Notebook**: interactive Python environment

![image.png](images/M6L1_Review_of_Computing_Concepts_52300363.png)

When we use RSSOndemand, we are using a web-based access portal to the Hellbender HPC. This provides a simple-to-use GUI(Graphical User Interface) for interacting with the cluster. These tools run on the cluster and are streamed to our web browsers. 

**Stepwise, Ondemand does the following:** 
1. Submits a SLURM job to Hellbender by opening an app, for example, Jupyter Notebook, for use in this course
2. Requests compute resources (CPUs, RAM, time)
3. Starts a session on a compute node
4. Connects your browser to that session

### Step 1: Submits a SLURM job to Hellbender by opening an app, for example, Jupyter Notebook, for use in this course
We then see a UI webpage, allowing us to interact with these sessions. Files and Storage are the same for use across any of these sessions within RSSOnDemand; they all access the storage on the cluster.

## Step 2: Requests compute resources (CPUs, RAM, time)
To choose allocations for each session, or job, within the RSSOnDemand server, you need to think about your requirements: 
- **Nodes**: The number of Nodes needed for the use of the Job. We will almost always use a single node, as our workflows can use multiple CPU cores for parallelization of tasks; we do not need multiple nodes for distributed computing when running an RNA-Seq analysis.
- **CPU Cores**: Number of processor cores for your job
- **RAM**: The amount of memory needed for use of our job in GB
- **Partition**: The group of nodes selected for. The chosen partition tells the server which group of nodes to allocate resources from. 

| Partition Name                                      | Time Limit (default / max)  | Main Intended Use / Notes                                                                                                                                                                          |
| --------------------------------------------------- | --------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **general**                                         | default 1 hour, max 2 days  | Default partition for most non-special jobs. Use when you have typical analyses, moderate compute needs. ([docs.itrss.umsystem.edu][1])                                                            |
| **requeue**                                         | default 10 min, max 2 days  | Used for jobs that were requeued because they landed on an investor-owned node (see “priority/ investor partition”). Fallback queue for non-investor jobs. ([docs.itrss.umsystem.edu][1])          |
| **gpu**                                             | default 1 hour, max 2 days  | For jobs that require one or more GPUs. Only GPU-enabled nodes belong here (with Nvidia A100 cards). ([docs.itrss.umsystem.edu][1])                                                                |
| **interactive**                                     | default 1 hour, max 4 hours | For short interactive sessions — testing, debugging, or interactive tools (e.g. small-scale Jupyter, compiling code). Safer than running on login node. ([docs.itrss.umsystem.edu][1])             |
| **logical_cpu**                                     | default 1 hour, max 2 days  | For workloads that make use of hyperthreading / logical CPUs rather than physical cores (when that mode is beneficial). ([docs.itrss.umsystem.edu][1])                                             |
| **priority / investor (group-specific) partitions** | up to 28 days               | Reserved for research groups that have “invested” (paid) for priority or dedicated access. Useful for heavy workloads or large workflows needing guaranteed access. ([docs.itrss.umsystem.edu][1]) |

[1]: https://docs.itrss.umsystem.edu/pub/hpc/hellbender?utm_source=chatgpt.com "Hellbender [IT Research Support Solutions Wiki]"

A quick reference of tasks you may use and their requirements is below: 
#### Recommended Hellbender Node Types for RNA-Seq and ChIP-seq Tasks

| Task / Tool | Dataset Size | Node Type | Why This Node? | Notes |
|-------------|--------------|-----------|-----------------|-------|
| FastQC | Any | standard node | Low RAM and CPU usage | Very fast, no need for special nodes |
| MultiQC | Any | standard node | Minimal compute workload | Runs in seconds-minutes |
| TrimGalore / Cutadapt | RNA-Seq or ChIP-seq | standard node | Moderate CPU, low RAM | Good for preprocessing |
| HISAT2 Alignment | RNA-Seq | standard node | Efficient CPU usage, moderate RAM | Works fine on all standard nodes |
| STAR Alignment (Arabidopsis) | Small plant genome | **standard or highmem** | Needs more memory than HISAT2 | Highmem recommended if RAM > 20 GB needed |
| STAR Alignment (Human) | Large genome | **highmem node strongly recommended** | Requires ~30–60 GB RAM | Avoid standard nodes unless memory fits |
| STAR Genome Indexing | Any large genome | **highmem node** | One of the highest RAM tasks | May need >60 GB RAM |
| featureCounts | RNA-Seq or ChIP-seq | standard node | Low memory but can use CPUs | Fast on standard nodes |
| StringTie | RNA-Seq | standard node | Moderate RAM needs | Works well unless huge datasets |
| MACS2 | ChIP-seq | standard node | Single-threaded, low memory | No need for special nodes |
| DeepTools (bamCoverage) | RNA-Seq or ChIP-seq | standard node | CPU-parallel, modest RAM | Good balance on standard nodes |
| Salmon / Kallisto | RNA-Seq | standard node | Lightweight & parallel | No need for highmem |
| RStudio (DESeq2) | RNA-Seq count tables | standard node unless very large dataset | DESeq2 uses RAM, not CPUs | Highmem only if samples > 100+ |
| IGV / Visualization | Any | interactive node (standard) | Needs GUI, low RAM | Good for browsing data |
| Jupyter Notebook | Any | interactive node (standard) | Code execution & visualization | Start here for development |

### Steps 3 and 4: Usage of the OnDemand Sessions: 

#### Jupyter Notebook
A browser-based interactive Python environment running on a compute node. Ideal for prototyping, testing scripts, small analyses, and teaching. We use this often within the course, and there are tutorials in Module 0 if you need further instruction. 

#### Hellbender Desktop
full remote Linux desktop running on a compute node via the browser. Provides a graphical environment for tools that need a GUI, and for easier navigation for student use.

#### Code Server
A cloud-hosted version of VS Code running on a compute node. It is excellent for editing scripts, exploring directories, writing SLURM files, and interactive shell work. See the video below on how to use the Code server: 



<iframe width="560" height="315" src="https://www.youtube.com/embed/VqCgcpAypFQ?si=XlnnUiykU8Eiyjy2" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

----
## Terminal usage of the Hellbender Cluster - SLURM, SBatch, and Terminal via SSH Connections

### SSH Connections: 
SSH (Secure Shell) allows you to connect from your local computer to the Hellbender cluster securely. We use either a program like [PuTTY](https://the.earth.li/~sgtatham/putty/0.83/htmldoc/), or the native terminal (Ie; PowerShell in Windows, or Terminal in MacOS) to connect to the HPC cluster. 

**How to Connect**
```
# Linux / macOS
ssh username@<Hellbender login host, see the RSS documentation> 

# Windows (via PowerShell or Putty)
ssh username@<Hellbender login host, see the RSS documentation> 
```

*[Screenshot omitted from the public version (it showed account-specific details).]*


Replacing username with your campus sso username, enter your sso password. Logging in places you onto the login node.

*[Screenshot omitted from the public version (it showed account-specific details).]*

**Under no circumstances should you run your code on the login node.** YOU MUST REQUEST AN INTERACTIVE NODE TO USE THE TERMINAL OR SUBMIT A JOB VIA SBATCH OR SLURM SCHEDULER. If you are submitting a batch file, then your job will be redirected to a compute node to be computed.

However, if you are attempting use a GUI, ensure that you do not run your session on the login node (Example: username@hellbender-login-p1). Use an interactive session to be directed to a compute node to run your software. 

Tips:
- Use -Y or -X if you want to forward GUI applications (X11 forwarding).
- Use SSH keys for passwordless login if allowed by IT.
- Once connected, you are in the cluster’s login node.


#### Off Campus Logins

Our off campus logins use public key authentication only, password authentication is disabled for off campus users unless they are connected to the campus VPN. Please send your public key to RSS support (see the Hellbender documentation) with the subject of “Hellbender Public Key Access”. After setting up your client to use your key, you still use the host <Hellbender login host, see the RSS documentation> to connect, however now without need for using the VPN. 

#### SSH Keys

If you want to connect using SSH keys, either to avoid having to type in your password, or are wanting to connect from off campus without a VPN, you can add your SSH public key to Hellbender.

For Windows users, we recommend using MobaXterm https://mobaxterm.mobatek.net. Powershell, git bash, and putty are some other available options. Mac and Linux users can use the default terminal.
Generating an SSH Key

You can generate a new SSH key on your local machine. After you generate the key, you can add the public key to your account on Hellbender.

    Open terminal
    Paste the text below, replacing the email used in the example with your University email address.
```
ssh-keygen -t ed25519 -C "sso@example.com"
```

When you're prompted to “Enter a file in which to save the key”, you can press Enter to accept the default file location. Please note that if you created SSH keys previously, ssh-keygen may ask you to rewrite another key, in which case we recommend creating a custom-named SSH key. To do so, type the default file location and replace id_ALGORITHM with your custom key name.
```
# Windows
Enter file in which to save the key (/c/Users/USERNAME/.ssh/id_ALGORITHM):[Press enter]
# Mac
Enter a file in which to save the key (/Users/USERNAME/.ssh/id_ALGORITHM): [Press enter]
# Linux
Enter a file in which to save the key (~/.ssh/id_ALGORITHM):[Press enter]
```
At the prompt, type a secure passphrase.

#### Adding your SSH key

You may add your own SSH public key to your Hellbender account. You can also send the key to RSS support (see the Hellbender documentation) and we can add it to your Hellbender account. 
Copy the contents of your SSH public key, which is written to the file created in the Generating an SSH Key step.
```
# Windows
Your public key has been saved in /c/Users/USERNAME/.ssh/id_ALGORITHM.pub
# Mac
Your public key has been saved in /Users/USERNAME/.ssh/id_ALGORITHM.pub
# Linux
Your public key has been saved in ~/.ssh/id_ALGORITHM.pub
```
Add your public key to your account by appending it to your authorized_keys file on Hellbender

```
[sso@hellbender-login ~]$ vim ~/.ssh/authorized_keys
```

#### 2. SLURM Overview

SLURM is the job scheduler that manages all computation on Hellbender.

Key concepts:

- Job: A single unit of work submitted to the cluster.
- Node: Physical machine running your job.
- Partition: Queue of nodes with specific capabilities (e.g., general, interactive, gpu).
- Wall time: Maximum runtime your job is allowed.

Useful commands: 
```
# Check available nodes and partitions
sinfo

# View your running and pending jobs
squeue -u your_username

# Cancel a job
scancel <job_id>
```

#### 3. Submitting a Job: 
Using SLURM, you need to create a submission script to execute on the backend nodes, then use a command line utility to submit the script to the resource manager. See the file contents of a general submission script complete with comments. 

Example Script: 
```
#!/bin/bash
#SBATCH --job-name=RNAseq_STAR
#SBATCH --output=STAR_%j.out
#SBATCH --error=STAR_%j.err
#SBATCH --partition=general
#SBATCH --nodes=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00

# Load necessary modules
module load STAR/2.7.10a

# Run STAR alignment
STAR --genomeDir /project/genomes/Arabidopsis_index \
     --readFilesIn sample_R1.fastq.gz sample_R2.fastq.gz \
     --runThreadN 8 \
     --outFileNamePrefix output/sample_


```

Now you need to submit that batch file to the scheduler so that it will run when it is time

```
 sbatch myjob.sh
```
Check the job status with: 
```
squeue -u your_username
```

Cancel the job if needed: 
```
scancel <job_id>
```

You will see the output of sbatch after the job submission that will give you the job number, if you would like to monitor the status of your jobs you may do so with the squeue command. If you submit a job to the requeue partition you will receive a warning message like: 
```
sbatch: Warning, you are submitting a job the to the requeue partition. There is a chance that your job
will be preempted by priority partition jobs and have to start over from the beginning.
Submitted batch job 167
```

**Common SBATCH Directives** 
| Directive          | Valid Values                         | Description |
|-------------------|-------------------------------------|-------------|
| --job-name=        | string value, no spaces             | Sets the job name to something friendly, useful when examining the queue. |
| --ntasks=          | integer value                        | Sets the requested number of CPUs for the job. |
| --nodes=           | integer value                        | Sets the number of nodes you wish to use, useful if you want all tasks to land on one node. |
| --time=            | D-HH:MM:SS, HH:MM:SS                | Sets the allowed run time for the job. |
| --mail-type=       | begin, end, fail, requeue            | Sets when the scheduler should notify you about the job. By default, no email is sent. |
| --mail-user=       | email address                        | Sets the email address for job notifications. |
| --export=          | ALL, or specific variable names      | By default, Slurm exports current environment variables to the job. |
| --mem=             | integer value                        | Amount of memory in MB for the job. Default values usually suffice. |
| --mem-per-cpu=     | integer value                        | Memory per CPU in MB (default ~800MB, depends on queue). |
| --nice=            | integer value                        | Lowers job priority; higher nice number = lower priority. |
| --constraint=      | see sbatch man page                  | Constrains the job to nodes with specific features. |
| --gres=            | name:count                           | Reserve additional resources, e.g., `--gres=gpu:2` reserves 2 GPUs. |
| -p                 | partition_name                       | Optional; specifies a partition. Otherwise, job goes to highest-priority allowed partition. |

**Valid Constraints**

| Feature       | Description |
|---------------|------------|
| intel         | Node has Intel CPUs |
| amd           | Node has AMD CPUs |
| EDR           | Node has EDR (100 Gbit/sec) InfiniBand interconnect |
| gpu           | Node has GPU acceleration capabilities |
| cpucodename*  | Node is running the desired CPU codename, e.g., rome |

#### 4. Interactive Jobs via salloc or srun

Sometimes you need a temporary interactive terminal on a compute node, this will act as the terminal similar to Jupyter Notebook, or the Hellbender Desktop:
```
# Request an interactive session on a compute node
salloc --partition=interactive --nodes=1 --cpus-per-task=4 --mem=8G --time=01:00:00

# Launch a shell on the allocated node
srun --pty bash
```
You can now run commands as if you are logged in to a single compute node. Ideal for testing scripts or debugging batch jobs.

<iframe width="560" height="315" src="https://www.youtube.com/embed/nVYW7IQCjmo?si=uzdTxG0RR6IwGP4L" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>


## Enviornment managment and Lmod Usage

On HPC systems like Hellbender, software is installed centrally and often multiple versions exist. To manage which version you use and avoid conflicts, clusters rely on environment modules, with Lmod being the module system used on Hellbender. We can also create local enviornments using Conda or mamba as we have done in the past, but there are a ton of programs already installed for use in RNA-Seq and NGS data processing. This section will review usage and creation of local environments - as you would use on your own machine, and the use of modules in the HPC cluster. 

Two main ways to manage environments:

- Lmod modules (pre-installed, cluster-wide)
- Mamba / Conda environments (user-managed, flexible)


### Conda/Mamba Usage
Using Mamba on the Cluster

1. Mamba is a fast alternative to Conda for creating isolated Python environments.
Useful when you need:

- Specific Python packages or versions not available as modules.
- Reproducible workflows in an isolated environment.
- Want to produce enviornments you could use on a local machine as well

2.1. Install / Load Mamba

Often, Mamba is available as a module:

`module load mamba`


If not available, you can install in your home directory.

2.2. Create a New Environment
```
# Create environment with Python 3.11 and specific packages
mamba create -n rnaseq_env python=3.11 pandas numpy matplotlib
```
2.3. Activate Environment
```
mamba activate rnaseq_env
```
2.4. Deactivate Environment
```
mamba deactivate
```
2.5. Install Additional Packages
```
mamba install seaborn scikit-learn
```

2.6. Export / Reproduce Environment
```
# Export environment YAML
mamba env export > rnaseq_env.yml

# Recreate environment elsewhere
mamba env create -f rnaseq_env.yml
```
3. Integrating Mamba Environments with SBATCH Scripts
```
#!/bin/bash
#SBATCH --job-name=rnaseq_analysis
#SBATCH --output=rnaseq_%j.out
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --time=04:00:00
#SBATCH --partition=general

# Load Mamba module
module load mamba

# Activate your custom environment
mamba activate rnaseq_env

# Run Python analysis
python rnaseq_pipeline.py

```
Tip: Always load Mamba and activate the environment in your SBATCH script to ensure compute nodes have the correct packages.



### Lmod Usage
#### 1. What is Lmod?

- Lmod is a module system that allows you to load, unload, and switch software environments dynamically.

- Avoids conflicts between software versions and ensures reproducibility.

- Each module modifies environment variables like PATH, LD_LIBRARY_PATH, etc., so the software runs correctly.

#### 2. Common Lmod Commands
| Command                       | Description                                                                |
| ----------------------------- | -------------------------------------------------------------------------- |
| `module avail`                | Lists all available modules on the cluster.                                |
| `module list`                 | Lists modules currently loaded in your environment.                        |
| `module load <module_name>`   | Loads the module into your environment (e.g., `module load STAR/2.7.10a`). |
| `module unload <module_name>` | Unloads the module from your environment.                                  |
| `module swap <old> <new>`     | Replaces one module version with another.                                  |
| `module purge`                | Unloads all modules. Useful for resetting your environment.                |
| `module spider <module_name>` | Searches for module versions and dependencies.                             |
| `module help <module_name>`   | Displays information about the module.                                     |

#### 3. Lmod example usage: 
```
# Check available modules
module avail

# Load STAR for RNA-Seq alignment
module load STAR/2.7.10a

# Check loaded modules
module list

# Run STAR
STAR --genomeDir /project/genomes/Arabidopsis_index \
     --readFilesIn sample_R1.fastq.gz sample_R2.fastq.gz \
     --runThreadN 8 \
     --outFileNamePrefix output/sample_

# Unload STAR when done
module unload STAR/2.7.10a
```

#### 4. Lmod Tips

1. Always check available modules before starting: `module avail` or `module spider`

2. Load only the modules you need to avoid conflicts.

3. Use `module purge` at the start of scripts to ensure a clean environment.

4. Check module dependencies using module spider if unsure.

5. Include `module load` commands in your SBATCH scripts so the environment is correct when jobs run on compute nodes.

### Lmod Vs Mamba usage


| Tool              | Strengths                                                                                                                        | When to Use                                                                                                                                                                                  |
| ----------------- | -------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Lmod Modules**  | - Pre-installed, maintained by HPC admins<br>- Lightweight, no overhead<br>- Integrated with SLURM                               | - Use when the module exists (e.g., STAR, HISAT2, Bowtie2)<br>- Use in SBATCH scripts for reproducible jobs<br>- Preferred for multi-user HPC workflows                                      |
| **Mamba / Conda** | - Flexible, user-managed environments<br>- Can install packages not available as modules<br>- Fully isolated Python environments | - Use when required software/version is missing in Lmod<br>- Running Python pipelines (Snakemake, pandas, matplotlib)<br>- Reproducible analysis across different clusters or local machines |
